#### Global Band adjustment coefficients used in HLS:



| HLS Band Name | Oli band#   | MSI band#   | Slpope (a) | Intercept (b) |
|------|---|---|--|--|
|CA    | 1 | 1 | 1.005 | 2.09E-04
|BLUE  | 2 | 2 | 1.020 | 4.47E-03
|GREEN | 3 | 3 | 0.994 | 1.09E-03
|RED   | 4 | 4 | 1.017 | -1.04E-03
|NIR1  | 5 | 8A | 0.999 | 2.50E-04
|SWIR1 | 6 | 11 | 0.999 | 1.24E-04
|SWIR2 | 7 | 12 | 1.003 | 1.19E-03


In [ ]:
#regression is assuming the data are scaled 0-1, 
#so the intercept should be multiplied by 10000 for our data

hls_coeffs = np.array([[1,1,1.005,0.000209],
                      [2,2,1.020,0.00447],
                      [3,3,0.994,0.00109],
                      [4,4,1.017,0.00104],
                      [5,8,0.999,0.00025],
                      [6,11,0.999,0.000124],
                      [7,12,1.003,0.00119]])
#not used


In [ ]:
#structure to store regression coefficients for now (only supports a simple regression...)
oli_from_msi = {
#                'oli01':  {'msi_band':'msi01','params':([1,0]),'r_params':([0,0]),'hls_params':([1.005,0.000209])},
                'oli02_agm':  {'msi_band':'msi02_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.020,0.00447])},
                'oli03_agm' : {'msi_band':'msi03_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([0.994,0.00109])},
                'oli04_agm' : {'msi_band':'msi04_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.017,0.00104])},
#                'oli05_agm' : {'msi_band':'msi08_agm','params':([1,0])},'r_params':([0,0]),'hls_params':([0.999,0.00025]),
#                'oli06_agm' : {'msi_band':'msi11_agm','params':([1,0])},'r_params':([0,0]),'hls_params':([0.999,0.000124]),
                'oli07_agm' : {'msi_band':'msi12_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.003,0.00119])}     
                }

# steps are .... make a dictionary of the variable names, 
# determine the total number of cells
# convert each variable into a 1-d array
# determine where all variables have a value
# use a random number mask to index a subset of those
# take the indexed subsets as the arrays for the stack...


#make a dictionary stack of the relevant arrays if they exist in the dataset
varstack = {}
for oli_varname in oli_from_msi.keys():
    if oli_varname in ds.data_vars :
        msi_varname = oli_from_msi[oli_varname]['msi_band']
        if msi_varname in ds.data_vars:
            varstack[oli_varname] = ds[oli_varname]
            varstack[msi_varname] = ds[msi_varname]

if np.size(list(varstack.keys())) == 0:
    print('no variables found in the dataset for regression')


total_cells = np.size(ds[list(varstack.keys())[0]])
mask        = np.ones((total_cells),dtype='int8')   #'mask' will be an index

#convert to 1Xn vector 
for varname in varstack.keys():
    varstack[varname] = np.asarray(ds[varname]).reshape(total_cells)
    mask              = mask * varstack[varname]
    
#reduce to observations where all values are present
for varname in varstack.keys():
    varstack[varname] = varstack[varname][mask>0]
    
value_cells  = np.size(varstack[namelist[0]])      #this is how many valid observations we have
target_cells = 100000                              #this is how many valid observations we want...

#reduce to the desired number of samples
rmask        = (np.random.rand(value_cells))<target_cells/value_cells
for varname in varstack.keys():
    varstack[varname] = varstack[varname][rmask]

#Only implemented a simple linear regression here, could try multivariate but over-fitting beceomes an issue.

for oli_var in oli_from_msi.keys():               #why not use varstack keys?
    msi_var = oli_from_msi[oli_var]['msi_band']
    y = varstack[oli_var]
    A = np.ones(len(y))    #correstponds to the offset
    A = np.vstack((A,varstack[msi_var]))
    (oli_from_msi[oli_var]['params'][1],oli_from_msi[oli_var]['params'][0]),b,c,d = \
        sp.linalg.lstsq(A.T,y)   #returns intercept, slope, other..

    # reverse regression, msi from oli
    y = varstack[msi_var]
    A = np.ones(len(y))           #correstponds to the offset
    A = np.vstack((A,varstack[oli_var]))
    (oli_from_msi[oli_var]['r_params'][1],oli_from_msi[oli_var]['r_params'][0]),b,c,d = \
        sp.linalg.lstsq(A.T,y)   #returns intercept, slope, other..
    

    
    
if False:
    x = varstack[msi_var]
    y = varstack[oli_var]

    # run the inverse fit first 
    slope,intercept = np.polyfit(y,x,1)
    oli_from_msi[oli_var]['r_params'][0] = np.round(slope,3)
    oli_from_msi[oli_var]['r_params'][1] = np.round(intercept,3)

    slope,intercept = np.polyfit(x,y,1)
    oli_from_msi[oli_var]['params'][0] = np.round(slope,3)
    oli_from_msi[oli_var]['params'][1] = np.round(intercept,3)
    #alternately could use this, which generalises to more dimensinos but gives poor results
    #if more than one independent band is included....
    if False:
        A = np.ones(len(y))     #this is the first column of the matrix, corresponding to the offset.
        for nm in varstack.keys():
            if 'msi' in nm:
                A = np.vstack((A,varstack[nm]))
        solution = sp.linalg.lstsq(A.T,y)      #can use indexing to pulll some columns from A for simplification
        if test : print(solution)

#the dictionary oli_from_msi now contains the simple regression parameters

#Plot the results
if test or verbose:
    oli_var                  = 'oli03_agm'
    msi_var                  = oli_from_msi[oli_var]['msi_band']
    slope,intercept          = oli_from_msi[oli_var]['params']
    slope_hls,intercept_hls = oli_from_msi[oli_var]['hls_params'] 

    #poly1d_hls    = np.poly1d([hls_coeffs[3,2],hls_coeffs[3,3]*10000]) #Extract the hls coeff and scale the intercept

    x = varstack[msi_var]
    y = varstack[oli_var]

    # plot the results
    #plt.hist2d(x,y,bins=20,cmap='grey')
    plt.plot(x,y,'yo')
    plt.plot(x,slope*x+intercept,'b')
    plt.plot(x,slope_hls*x+intercept_hls, '--k')

    #, x, x*slope + intercept '--k',x, (x*slope_hls + intercept_hls) 'b') #'--k'=black dashed line, 'yo' = yellow circle marker
    plt.xlabel(msi_var)
    plt.ylabel(oli_var)

    plt.xlim(0, np.quantile(x,.99)*1.2)
    plt.ylim(0, np.quantile(y,.99)*1.2)

